[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/induction_ac.ipynb)

# Electromagnetic induction and alternating current full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: a magnet passing through a coil, a magnet falling down a
metal pipe, a rod coasting on rails, a current growing through an inductor, peak and rms values of three wave shapes,
the lead and lag of the current in an inductor and a capacitor, a series LCR circuit with its resonance curve, and a
power line. You can move all of them yourself: the magnet's **speed**, the pipe's **metal**, the rod's **field**, the
coil's **inductance**, the wave's **shape**, the circuit's **resistance** and **frequency**, and the line's **voltage**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `4e-7*pi` for
4π × 10⁻⁷, `2e6` for 2 × 10⁶ (`2*10^6` works too). You can type the arithmetic: `2*pi*50*0.2`, `170/sqrt(2)`,
`0.8*0.03*cosd(30)*1000`, `1/(2*pi*sqrt(0.02*5e-7))`. Numbers without thousands separators: `1591.55`, not `1,591.55`.
For several answers, separate them with commas, in the order the question lists them (the box's label shows the
order). You can also put your answer between the quotes in a cell's last line, `practice("P1.3", "")`, and run the
cell: it checks straight away.

**The cheat sheet.** Every formula, direction and constant from the course on two pages:
[induction and alternating current cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/induction_ac_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # wires and waves in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# current and speed SKY · force CORAL · energy and power VIOLET · voltage and flux CHALK · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
MU0 = 4 * math.pi * 1e-7   # T m/A, mu nought: four pi times ten to the minus seven
g = 9.8                    # m/s²
ROOT2 = math.sqrt(2.0)

# ---- the course's simulator: fourth-order Runge–Kutta, a fixed step at a time ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """deriv(t, s) -> ds/dt as a tuple; state a tuple of numbers. Returns the list of (t, state) samples from 0 to
    t_end, or up to the first step where stop(t, s) is True."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t = t + dt
        out.append((t, s))
        if stop is not None and stop(t, s): return out
    return out

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def sind(d): return math.sin(math.radians(d))
def cosd(d): return math.cos(math.radians(d))
def flux(B, A, theta=0.0): return B * A * cosd(theta)                                # Φ = B A cos θ
def emf_rate(N, dphi, dt): return N * dphi / dt                                      # ε = N ΔΦ/Δt (its size)
def motional(B, l, v): return B * l * v                                              # ε = B l v
def mutual_solenoids(n1, n2, A, l): return MU0 * n1 * n2 * A * l                     # M = μ₀ n₁ n₂ A l
def self_solenoid(n, A, l): return MU0 * n * n * A * l                               # L = μ₀ n² A l
def inductor_energy(L, I): return 0.5 * L * I * I                                    # U = ½ L I²
def omega(f): return 2 * math.pi * f                                                 # ω = 2π f
def gen_peak(N, B, A, f): return N * B * A * omega(f)                                # ε₀ = N B A ω
def rms(peak_): return peak_ / ROOT2                                                 # rms = peak/√2
def peak(rms_): return rms_ * ROOT2
def x_l(f, L): return omega(f) * L                                                   # X_L = ω L
def x_c(f, C): return 1.0 / (omega(f) * C)                                           # X_C = 1/(ω C)
def impedance(R, XL=0.0, XC=0.0): return math.hypot(R, XL - XC)                      # Z = √(R² + (X_L − X_C)²)
def phase_deg(R, XL=0.0, XC=0.0): return math.degrees(math.atan2(XL - XC, R))        # tan φ = (X_L − X_C)/R
def resonant_omega(L, C): return 1.0 / math.sqrt(L * C)                              # ω₀ = 1/√(LC)
def resonant_f(L, C): return resonant_omega(L, C) / (2 * math.pi)
def q_factor(L, C, R): return resonant_omega(L, C) * L / R                           # Q = ω₀ L/R
def ac_power(V, I, phi=0.0): return V * I * cosd(phi)                                # P = V I cos φ (rms values)
def secondary_voltage(Vp, Np, Ns): return Vp * Ns / Np                               # V_s/V_p = N_s/N_p
def line_loss(P, V, R): return (P / V) ** 2 * R                                      # loss = (P/V)² R
def dipole_flux(m, a, d): return MU0 * m * a * a / (2 * (a * a + d * d) ** 1.5)      # a small magnet's flux through a loop

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cos": math.cos, "sin": math.sin, "acos": math.acos, "atan": math.atan,
         "cosd": cosd, "sind": sind}

def calc(text):
    """Plain arithmetic only: numbers (4e-7 too), + − × ÷, powers, brackets, sqrt, cos, sin, acos, atan (radians), cosd
    and sind (degrees) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(mWb|Wb|mH|H|kHz|Hz|rad/s|mV|kV|V|mA|A|mN|N|mJ|J|kW|MW|W|Ω|ohms?|nF|µF|uF|mC|C|ms|s|cm|mm|m|T|%|°|"
        r"degrees?|turns?|times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": "Flux and Faraday's law",
     "q": 'A magnetic field of 0.3 T passes straight through a loop of area 0.04 m² (face-on). What is the flux through the loop, in milliweber?',
     "parts": ['flux (mWb)'],
     "answer": [12.0], "tol": [0.005], "function": '—',
     "hint": 'face-on, cos θ = 1: Φ = B A',
     "working": 'Φ = BA = 0.3 × 0.04 = 0.012 Wb = 12 mWb'},
    {"id": 'P1.2', "tier": 1, "group": "Flux and Faraday's law",
     "q": 'The flux through a single loop falls by 0.06 Wb in 0.2 s, at a steady rate. What is the size of the induced EMF, in volts?',
     "parts": ['EMF (V)'],
     "answer": [0.3], "tol": [0.005], "function": '—',
     "hint": "Faraday's law: the EMF is the rate of change of flux",
     "working": 'ε = ΔΦ/Δt = 0.06 / 0.2 = 0.3 V'},
    {"id": 'P1.3', "tier": 1, "group": 'Motional EMF',
     "q": 'A rod 0.5 m long moves at 6 m/s at right angles to a field of 0.4 T (and to its own length). What is the EMF between its ends, in volts?',
     "parts": ['EMF (V)'],
     "answer": [1.2000000000000002], "tol": [0.005], "function": '—',
     "hint": 'ε = B l v',
     "working": 'ε = Blv = 0.4 × 0.5 × 6 = 1.2 V'},
    {"id": 'P1.4', "tier": 1, "group": 'Inductance and the generator',
     "q": 'The current in a 0.2 H inductor rises at 30 A/s. What is the size of the back EMF, in volts?',
     "parts": ['EMF (V)'],
     "answer": [6.0], "tol": [0.005], "function": '—',
     "hint": 'ε = L ΔI/Δt',
     "working": 'ε = L × (rate of change of current) = 0.2 × 30 = 6 V'},
    {"id": 'P1.5', "tier": 1, "group": 'Inductance and the generator',
     "q": 'How much energy is stored in a 0.4 H inductor carrying 5 A, in joules?',
     "parts": ['energy (J)'],
     "answer": [5.0], "tol": [0.005], "function": '—',
     "hint": 'U = ½ L I²',
     "working": 'U = ½LI² = 0.5 × 0.4 × 25 = 5 J'},
    {"id": 'P1.6', "tier": 1, "group": 'AC, rms and reactance',
     "q": 'An alternating voltage has a peak value of 170 V. What is its rms value, in volts?',
     "parts": ['rms voltage (V)'],
     "answer": [120.20815280171307], "tol": [0.005], "function": 'sqrt',
     "hint": 'rms = peak ÷ √2',
     "working": 'V_rms = V_peak/√2 = 170 / 1.4142 = 120.2082 V'},
    {"id": 'P1.7', "tier": 1, "group": 'AC, rms and reactance',
     "q": 'What is the reactance of a 0.2 H inductor at 50 Hz, in ohms?',
     "parts": ['reactance (Ω)'],
     "answer": [62.83185307179587], "tol": [0.005], "function": 'pi',
     "hint": 'X_L = ωL, with ω = 2πf',
     "working": 'X_L = 2πfL = 2π × 50 × 0.2 = 62.8319 Ω'},
    {"id": 'P1.8', "tier": 1, "group": 'AC, rms and reactance',
     "q": 'What is the reactance of a 20 µF capacitor at 50 Hz, in ohms?',
     "parts": ['reactance (Ω)'],
     "answer": [159.15494309189532], "tol": [0.005], "function": 'pi',
     "hint": 'X_C = 1/(ωC), with C in farads',
     "working": 'X_C = 1/(2πfC) = 1 / (2π × 50 × 20 × 10⁻⁶) = 159.1549 Ω'},
    {"id": 'P1.9', "tier": 1, "group": 'LCR circuits',
     "q": 'A 6 Ω resistor is in series with an inductor whose reactance is 8 Ω. What is the impedance, in ohms?',
     "parts": ['impedance (Ω)'],
     "answer": [10.0], "tol": [0.005], "function": 'sqrt',
     "hint": 'Z = √(R² + (X_L − X_C)²), here with no capacitor',
     "working": 'Z = √(R² + X_L²) = √(36 + 64) = 10 Ω'},
    {"id": 'P1.10', "tier": 1, "group": 'Transformers and power lines',
     "q": 'An ideal transformer has 1000 turns on its primary and 50 on its secondary. The primary is on 230 V. What is the secondary voltage, in volts?',
     "parts": ['secondary voltage (V)'],
     "answer": [11.5], "tol": [0.005], "function": '—',
     "hint": 'V_s/V_p = N_s/N_p',
     "working": 'V_s = V_p × N_s/N_p = 230 × 50/1000 = 11.5 V'},
    {"id": 'P2.1', "tier": 2, "group": "Flux and Faraday's law",
     "q": 'A coil of 300 turns, each of area 20 cm², faces a field that rises steadily from 0.1 T to 0.5 T in 0.02 s. What is the size of the induced EMF, in volts?',
     "parts": ['EMF (V)'],
     "answer": [12.0], "tol": [0.005], "function": '—',
     "hint": '20 cm² is 0.002 m²; ε = N A ΔB/Δt',
     "working": 'ΔΦ per turn = A ΔB = 0.002 × 0.4 = 8 × 10⁻⁴ Wb; ε = N ΔΦ/Δt = 300 × 8 × 10⁻⁴ / 0.02 = 12 V'},
    {"id": 'P2.2', "tier": 2, "group": "Flux and Faraday's law",
     "q": 'A loop 0.15 m by 0.2 m sits in a field of 0.8 T. The line square to its face makes 30° with the field. What is the flux through it, in milliweber?',
     "parts": ['flux (mWb)'],
     "answer": [20.78460969082653], "tol": [0.005], "function": 'cos',
     "hint": 'θ is measured from the line square to the face: Φ = B A cos θ',
     "working": 'Φ = BA cos θ = 0.8 × 0.03 × cos 30° = 0.020785 Wb = 20.7846 mWb'},
    {"id": 'P2.3', "tier": 2, "group": 'Motional EMF',
     "q": "A rod 0.6 m long slides at 5 m/s on rails through a field of 0.5 T; the circuit's resistance is 3 Ω. What current flows, in amps, and what force holds the rod back, in newtons?",
     "parts": ['current (A)', 'force (N)'],
     "answer": [0.5, 0.15], "tol": [0.005, 0.005], "function": '—',
     "hint": 'ε = B l v, then I = ε/R, then F = B I l',
     "working": 'ε = Blv = 0.5 × 0.6 × 5 = 1.5 V; I = ε/R = 0.5 A; F = BIl = 0.5 × 0.5 × 0.6 = 0.15 N'},
    {"id": 'P2.4', "tier": 2, "group": 'Inductance and the generator',
     "q": 'A solenoid has 800 turns, is 0.4 m long and has a cross-section of 5 cm². What is its self-inductance, in millihenry?',
     "parts": ['self-inductance (mH)'],
     "answer": [1.0053096491487339], "tol": [0.005], "function": 'pi',
     "hint": 'L = μ₀ n² A l, with n the turns per metre',
     "working": 'n = 800/0.4 = 2000 per metre; L = μ₀n²Al = 4π × 10⁻⁷ × 2000² × 5 × 10⁻⁴ × 0.4 = 1.0053 mH'},
    {"id": 'P2.5', "tier": 2, "group": 'Inductance and the generator',
     "q": "A generator's coil has 150 turns of area 0.02 m² and turns 60 times a second in a field of 0.3 T. What is the peak EMF, and what is its rms value, both in volts?",
     "parts": ['peak EMF (V)', 'rms EMF (V)'],
     "answer": [339.29200658769764, 239.91567866055175], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'ε₀ = N B A ω, with ω = 2πf; then divide by √2',
     "working": 'ω = 2π × 60 = 376.9911 rad/s; ε₀ = NBAω = 150 × 0.3 × 0.02 × 376.9911 = 339.2920 V; rms = ε₀/√2 = 239.9157 V'},
    {"id": 'P2.6', "tier": 2, "group": 'AC, rms and reactance',
     "q": 'A 46 Ω heater runs on 230 V rms. What are the rms current in amps, the average power in watts, and the peak current in amps?',
     "parts": ['rms current (A)', 'power (W)', 'peak current (A)'],
     "answer": [5.0, 1150.0, 7.0710678118654755], "tol": [0.005, 0.005, 0.005], "function": 'sqrt',
     "hint": 'use rms values for the power; peak = rms × √2',
     "working": 'I_rms = 230/46 = 5 A; P = V_rms I_rms = 230 × 5 = 1150 W; I_peak = 5 × √2 = 7.0711 A'},
    {"id": 'P2.7', "tier": 2, "group": 'AC, rms and reactance',
     "q": 'A 40 µF capacitor is joined to a 120 V rms, 60 Hz supply. What is its reactance in ohms, and what rms current flows, in amps?',
     "parts": ['reactance (Ω)', 'rms current (A)'],
     "answer": [66.31455962162306, 1.8095573684677209], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'X_C = 1/(ωC), then I = V/X_C',
     "working": 'X_C = 1/(2π × 60 × 40 × 10⁻⁶) = 66.3146 Ω; I = V/X_C = 120 / 66.3146 = 1.8096 A'},
    {"id": 'P2.8', "tier": 2, "group": 'LCR circuits',
     "q": 'A 50 Ω resistor, a 0.3 H inductor and a 50 µF capacitor are in series on a 230 V rms, 50 Hz supply. What is the impedance in ohms, and what rms current flows, in amps?',
     "parts": ['impedance (Ω)', 'rms current (A)'],
     "answer": [58.613064300324154, 3.924039849230814], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'find X_L and X_C first; Z = √(R² + (X_L − X_C)²)',
     "working": 'X_L = 2π × 50 × 0.3 = 94.2478 Ω; X_C = 1/(2π × 50 × 50 × 10⁻⁶) = 63.6620 Ω; Z = √(50² + (94.25 − 63.66)²) = 58.6131 Ω; I = 230/Z = 3.9240 A'},
    {"id": 'P2.9', "tier": 2, "group": 'LCR circuits',
     "q": 'At what frequency, in hertz, does a 20 mH inductor resonate with a 0.5 µF capacitor?',
     "parts": ['resonant frequency (Hz)'],
     "answer": [1591.5494309189535], "tol": [0.005], "function": 'pi',
     "hint": 'ω₀ = 1/√(LC), and f₀ = ω₀/(2π)',
     "working": 'f₀ = 1/(2π√(LC)) = 1 / (2π × √(0.02 × 5 × 10⁻⁷)) = 1591.5494 Hz'},
    {"id": 'P2.10', "tier": 2, "group": 'Transformers and power lines',
     "q": 'A line of resistance 4 Ω carries 50 kW at 5,000 V. How much power is lost in the line, in watts, and what percentage of the 50 kW is that?',
     "parts": ['loss (W)', 'loss (%)'],
     "answer": [400.0, 0.8], "tol": [0.005, 0.005], "function": '—',
     "hint": 'the current is P/V; the loss is I²R',
     "working": 'I = P/V = 50,000/5,000 = 10 A; loss = I²R = 100 × 4 = 400 W; 400/50,000 = 0.8 %'},
    {"id": 'P3.1', "tier": 3, "group": "Flux and Faraday's law",
     "q": 'A 250-turn coil of area 40 cm² is turned from face-on to edge-on in 0.05 s, and the average EMF is 3 V. How strong is the field, in tesla?',
     "parts": ['field (T)'],
     "answer": [0.15000000000000002], "tol": [0.005], "function": '—',
     "hint": 'edge-on, the flux is zero: ΔΦ = B A for each turn',
     "working": 'the flux through each turn falls from BA to 0, so ε = NBA/Δt and B = εΔt/(NA) = 3 × 0.05 / (250 × 0.004) = 0.15 T'},
    {"id": 'P3.2', "tier": 3, "group": 'Motional EMF',
     "q": "A 0.2 kg rod, 0.4 m long, slides on rails in a 0.5 T field; the circuit's resistance is 1 Ω. It is given 3 m/s and let go. What is its time constant mR/(Bl)² in seconds, how much heat is made in all, in joules, and how far does it travel, in metres?",
     "parts": ['time constant (s)', 'heat (J)', 'distance (m)'],
     "answer": [4.999999999999999, 0.9, 15.0], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'the force (Bl)²v/R is always against the motion; energy is conserved; distance = v₀ τ',
     "working": 'τ = mR/(Bl)² = 0.2 × 1 / (0.2)² = 5 s; all the energy of motion becomes heat: ½mv² = 0.5 × 0.2 × 9 = 0.9 J; the speed dies away as e^(−t/τ), so the distance is vτ = 3 × 5 = 15 m'},
    {"id": 'P3.3', "tier": 3, "group": 'Motional EMF',
     "q": "A 100-turn coil of area 10 cm² lies face-on to a 0.2 T field and is flipped right over. The circuit's resistance is 20 Ω. How much charge flows, in millicoulombs? (Charge = current × time, so it is N ΔΦ/R, however fast the flip.)",
     "parts": ['charge (mC)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'ε = N ΔΦ/Δt and I = ε/R, so q = I Δt = N ΔΦ/R: the time cancels',
     "working": 'flipping reverses the flux: ΔΦ = 2BA = 2 × 0.2 × 0.001 = 4 × 10⁻⁴ Wb per turn; q = NΔΦ/R = 100 × 4 × 10⁻⁴ / 20 = 2 × 10⁻³ C = 2 mC'},
    {"id": 'P3.4', "tier": 3, "group": 'Inductance and the generator',
     "q": 'Two coils are wound on one tube 0.3 m long, of cross-section 8 cm²: one with 1500 turns per metre, the other with 3000. What is their mutual inductance, in millihenry? The current in the first falls from 2 A to zero in 0.004 s: what EMF appears in the second, in volts?',
     "parts": ['mutual inductance (mH)', 'EMF (V)'],
     "answer": [1.3571680263507904, 0.6785840131753952], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'M = μ₀ n₁ n₂ A l; then ε = M × (rate of change of current)',
     "working": 'M = μ₀n₁n₂Al = 4π × 10⁻⁷ × 1500 × 3000 × 8 × 10⁻⁴ × 0.3 = 1.3572 mH; ε = M ΔI/Δt = 0.001357 × 500 = 0.6786 V'},
    {"id": 'P3.5', "tier": 3, "group": 'AC, rms and reactance',
     "q": 'An inductor with no resistance draws 0.46 A rms from a 230 V rms, 50 Hz supply. What is its inductance, in henry?',
     "parts": ['inductance (H)'],
     "answer": [1.5915494309189533], "tol": [0.005], "function": 'pi',
     "hint": 'the reactance is V/I; then X_L = 2πfL run backwards',
     "working": 'X_L = V/I = 230/0.46 = 500 Ω; L = X_L/(2πf) = 500 / (2π × 50) = 1.5915 H'},
    {"id": 'P3.6', "tier": 3, "group": 'LCR circuits',
     "q": 'A 20 Ω resistor, a 0.1 H inductor and a 100 µF capacitor are in series on a 230 V rms, 60 Hz supply. What is the power factor, and what average power is delivered, in watts?',
     "parts": ['power factor', 'power (W)'],
     "answer": [0.8730026337490993, 2015.8433681194247], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'cos φ = R/Z; only the resistor takes power: P = I²R = V I cos φ',
     "working": 'X_L = 37.6991 Ω, X_C = 26.5258 Ω, Z = 22.9094 Ω; cos φ = R/Z = 0.8730; I = 230/Z = 10.0395 A; P = I²R = 2015.84 W (the same as V I cos φ)'},
    {"id": 'P3.7', "tier": 3, "group": 'LCR circuits',
     "q": 'A circuit with a 2 mH inductor must resonate at 100 kHz with a quality factor of 50. What capacitance does it need, in nanofarads, and what is the most resistance it can have, in ohms?',
     "parts": ['capacitance (nF)', 'resistance (Ω)'],
     "answer": [1.2665147955292224, 25.13274122871834], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'ω₀ = 1/√(LC) run backwards, then Q = ω₀L/R run backwards',
     "working": 'C = 1/(ω₀²L) = 1 / ((2π × 10⁵)² × 0.002) = 1.2665 nF; R = ω₀L/Q = 1256.64 / 50 = 25.1327 Ω'},
    {"id": 'P3.8', "tier": 3, "group": 'Transformers and power lines',
     "q": 'A 3 MW wind turbine generates at 690 V. An ideal transformer raises this to 66 kV for a cable of resistance 0.8 Ω (treat it as one circuit). What is the turns ratio, what current flows in the cable in amps, and how much power does the cable lose, in watts?',
     "parts": ['turns ratio', 'cable current (A)', 'loss (W)'],
     "answer": [95.65217391304348, 45.45454545454545, 1652.892561983471], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'N_s/N_p = V_s/V_p; the cable carries P/V at the high voltage; loss = I²R',
     "working": 'turns ratio = 66,000/690 = 95.6522; I = P/V = 3,000,000/66,000 = 45.4545 A; loss = I²R = 45.4545² × 0.8 = 1652.89 W'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert abs(MU0 / (4 * math.pi) - 1e-7) < 1e-20
print("Ready: mu nought =", MU0, "T m/A; root two =", round(ROOT2, 4))

## Tier 1 · Warm-up

One idea, one step: the flux through a loop, Faraday's law, a moving rod's EMF, an inductor's back EMF and its
energy, an rms value, the reactance of an inductor and of a capacitor, an impedance, and a transformer's voltage.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `flux(B, A, theta)`, `emf_rate(N, dphi, dt)`,
`motional(B, l, v)`, `inductor_energy(L, I)`, `rms(peak)`, `x_l(f, L)`, `x_c(f, C)`, `impedance(R, XL, XC)`,
`secondary_voltage(Vp, Np, Ns)` and the rest. Work in webers, henries, farads, ohms and hertz (mWb is `1e-3`, µF is
`1e-6`, cm² is `1e-4`), then change the unit at the end. Each line below computes one warm-up answer and checks it
against the practice set.

In [ ]:
print(f"P1.7  X_L = {x_l(50, 0.2):.2f} Ω;  P1.8  X_C = {x_c(50, 20e-6):.2f} Ω")
assert agrees("P1.1", [flux(0.3, 0.04) * 1e3]) and agrees("P1.2", [emf_rate(1, 0.06, 0.2)])
assert agrees("P1.3", [motional(0.4, 0.5, 6)]) and agrees("P1.4", [0.2 * 30])
assert agrees("P1.5", [inductor_energy(0.4, 5)]) and agrees("P1.6", [rms(170)])
assert agrees("P1.7", [x_l(50, 0.2)]) and agrees("P1.8", [x_c(50, 20e-6)])
assert agrees("P1.9", [impedance(6, 8)]) and agrees("P1.10", [secondary_voltage(230, 1000, 50)])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## A magnet passing through a coil

A small strong magnet travels along the axis of a coil and out the other side. The upper graph is the flux through
one turn: it rises as the magnet comes in, peaks as it crosses the coil, and falls as it leaves. The lower graph is
the EMF, measured here as minus the slope of the flux between neighbouring points, times the number of turns. The EMF
goes one way coming in, passes through zero at the flux's peak (a flat slope), and goes the other way going out.
Double the speed: the flux curve keeps its height, but the EMF doubles. It is the rate of change that counts.

In [ ]:
def pass_run(v, turns, m=1.4, a=0.02, span=0.12, n=801):
    xs = np.linspace(-span, span, n); ts = (xs + span) / v
    ph = np.array([dipole_flux(m, a, abs(x)) for x in xs])
    em = np.zeros(n); em[1:-1] = -turns * (ph[2:] - ph[:-2]) / (ts[2:] - ts[:-2])      # minus the measured slope
    return ts, ph, em

def pass_picture(speed=1.0, turns=100):
    ts, ph, em = pass_run(speed, turns)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
    fig.subplots_adjust(left=0.1, right=0.98, top=0.88, bottom=0.11, hspace=0.18)
    a1.plot(ts * 1e3, ph * 1e6, color=CHALK); a1.set_ylabel("flux in one turn (µWb)")
    a2.plot(ts * 1e3, em * 1e3, color=GOLD); a2.set_ylabel("EMF (mV)"); a2.set_xlabel("time (ms)")
    fig.suptitle(f"flux peak {ph.max() * 1e6:.1f} µWb · EMF from {em.min() * 1e3:.0f} to {em.max() * 1e3:.0f} mV",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "the EMF is the measured slope of the flux"); plt.show()
    return ph.max(), em.max(), em[len(em) // 2]

f1, e1, mid1 = pass_picture(1.0, 100)
f2, e2, _ = pass_picture(2.0, 100)
assert close(f2, f1, 1e-12) and close(e2, 2 * e1, 1e-9) and abs(mid1) < 1e-12      # the same flux, twice the EMF; zero at the peak
slope_max = 3 * MU0 * 1.4 * 0.02 ** 2 * 0.01 / (2 * (1.25 * 0.02 ** 2) ** 2.5)     # the steepest slope, from the formula
assert close(e1, 100 * slope_max * 1.0, 2e-3)

In [ ]:
interact(pass_picture, speed=FloatSlider(value=1.0, min=0.25, max=4.0, step=0.25, description="speed (m/s)"),
         turns=IntSlider(value=100, min=10, max=500, step=10, description="turns"));

## A magnet falling down a metal pipe

The pipe is not magnetic, but it conducts. Each ring of the pipe has the magnet's flux changing through it, so it
carries a current; by Lenz's law the rings below push the magnet up and the rings above pull it up. The drag grows
with the speed, so the magnet soon falls at a steady speed where the drag equals its weight. Here the drag is added up
ring by ring, and the fall is stepped. Change the metal (copper conducts better than aluminium) and the wall's
thickness, and compare with free fall. This is our own model: a small magnet in a thin-walled pipe.

In [ ]:
METALS = {"copper": 5.96e7, "aluminium": 3.5e7, "plastic": 0.0}                    # conductivity, S/m, at 20 °C

def pipe_drag(sigma, wall, a=0.010, m=1.4, half=0.25, dz=1e-4):
    z = (np.arange(int(2 * half / dz)) + 0.5) * dz - half
    dphi = 3 * MU0 * m * a * a * z / (2 * (a * a + z * z) ** 2.5)                    # how fast each ring's flux changes, per m/s
    return float(np.sum(dphi ** 2) * sigma * wall * dz / (2 * math.pi * a))          # force = k × speed

def pipe_picture(metal="copper", wall_mm=1.0, mass_g=10.0, length=1.0):
    k = pipe_drag(METALS[metal], wall_mm * 1e-3); mass = mass_g * 1e-3
    run = simulate(lambda t, s: (s[1], g - k / mass * s[1]), (0.0, 0.0), 30.0, 2e-3, stop=lambda t, s: s[0] >= length)
    ts = np.array([t for t, _ in run]); depth = np.array([s[0] for _, s in run])
    t_free = math.sqrt(2 * length / g)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.16)
    tf = np.linspace(0, t_free, 50)
    ax.plot(tf, 0.5 * g * tf ** 2, color=DIM, label="free fall"); ax.plot(ts, depth, color=GOLD, label=f"in the {metal} pipe")
    ax.invert_yaxis(); ax.set_xlabel("time (s)"); ax.set_ylabel("depth (m)"); ax.legend(frameon=False)
    v_end = mass * g / k if k > 0 else float("inf")
    fig.suptitle(f"{length:g} m in {ts[-1]:.2f} s (free fall {t_free:.2f} s)" +
                 (f" · steady speed {v_end * 100:.1f} cm/s" if k > 0 else ""), x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own model · the drag added up ring by ring"); plt.show()
    return ts[-1], run[-1][1][1], k

t_cu, v_cu, k_cu = pipe_picture("copper", 1.0)
assert close(k_cu, 45 * MU0 ** 2 * 1.4 ** 2 * 5.96e7 * 1e-3 / (1024 * 0.010 ** 4), 1e-3)      # the sum against the formula
assert close(v_cu, 0.010 * g / k_cu, 1e-3) and 8.0 < t_cu < 8.6
t_al, _, _ = pipe_picture("aluminium", 1.0)
assert t_al < t_cu                                                                    # a poorer conductor brakes less

In [ ]:
interact(pipe_picture, metal=Dropdown(options=list(METALS), value="copper", description="metal"),
         wall_mm=FloatSlider(value=1.0, min=0.5, max=3.0, step=0.5, description="wall (mm)"),
         mass_g=FloatSlider(value=10.0, min=5.0, max=40.0, step=5.0, description="magnet (g)"),
         length=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description="pipe (m)"));

## Tier 2 · Core

The level of the course's worked problems: a coil in a changing field, a tilted loop, a rod on rails with its current
and force, a solenoid's inductance, a generator, a heater on AC, a capacitor's current, a series LCR circuit, a
resonant frequency, and a power line.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

For a tilted loop, the angle is measured from the line square to its face. For a rod on rails: the EMF, then the
current, then the force. For an LCR circuit, find the two reactances first; they pull opposite ways, so subtract them
before using Pythagoras.

In [ ]:
assert agrees("P2.1", [emf_rate(300, 20e-4 * (0.5 - 0.1), 0.02)]) and agrees("P2.2", [flux(0.8, 0.15 * 0.2, 30) * 1e3])
emf = motional(0.5, 0.6, 5); current = emf / 3
assert agrees("P2.3", [current, 0.5 * current * 0.6])
assert agrees("P2.4", [self_solenoid(800 / 0.4, 5e-4, 0.4) * 1e3])
assert agrees("P2.5", [gen_peak(150, 0.3, 0.02, 60), rms(gen_peak(150, 0.3, 0.02, 60))])
assert agrees("P2.6", [230 / 46, 230 * 230 / 46, peak(230 / 46)])
assert agrees("P2.7", [x_c(60, 40e-6), 120 / x_c(60, 40e-6)])
Z = impedance(50, x_l(50, 0.3), x_c(50, 50e-6))
print(f"P2.8  X_L = {x_l(50, 0.3):.2f} Ω, X_C = {x_c(50, 50e-6):.2f} Ω, Z = {Z:.2f} Ω, I = {230 / Z:.2f} A")
assert agrees("P2.8", [Z, 230 / Z]) and agrees("P2.9", [resonant_f(20e-3, 0.5e-6)])
assert agrees("P2.10", [line_loss(50e3, 5000, 4), 100 * line_loss(50e3, 5000, 4) / 50e3])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## A rod coasting on rails

A rod on two rails, in a field, is given a shove and let go. Its motion makes an EMF, B l v; the EMF sends a current
through the resistor; and the field pushes back on that current with a force B I l, always against the motion. So the
rod slows. The bars show where the energy goes: what the rod loses as energy of motion, the resistor gains as heat,
joule for joule. A stronger field or a smaller resistance brakes it sooner.

In [ ]:
def coast_picture(B=0.2, R=2.0, l=0.5, mass=0.1, v0=4.0):
    k = (B * l) ** 2 / R
    tau = mass / k
    run = simulate(lambda t, s: (s[1], -k / mass * s[1], k * s[1] * s[1]), (0.0, v0, 0.0), 5 * tau, tau / 400)
    ts = np.array([t for t, _ in run]); v = np.array([s[1] for _, s in run]); heat = np.array([s[2] for _, s in run])
    ke = 0.5 * mass * v ** 2
    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.16)
    ax.plot(ts, ke, color=SKY, label="energy of motion"); ax.plot(ts, heat, color=VIOLET, label="heat in the resistor")
    ax.plot(ts, ke + heat, color=GOLD, ls="--", label="their sum")
    ax.set_xlabel("time (s)"); ax.set_ylabel("energy (J)"); ax.legend(frameon=False)
    fig.suptitle(f"time constant m R/(B l)² = {tau:.1f} s · it travels {v0 * tau:.1f} m · heat in the end {0.5 * mass * v0 ** 2:.2f} J",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return tau, v[400] / v0, ke[-1] + heat[-1], run[-1][1][0]

tau_c, ratio_c, total_c, x_c5 = coast_picture(0.2, 2.0)
assert close(tau_c, 20.0, 1e-9) and close(ratio_c, math.exp(-1), 1e-6) and close(total_c, 0.8, 1e-6)
tau_p, _, total_p, _ = coast_picture(0.5, 1.0, 0.4, 0.2, 3.0)                     # exercise P3.2's rod
assert agrees("P3.2", [tau_p, total_p, 3.0 * tau_p])

In [ ]:
interact(coast_picture, B=FloatSlider(value=0.2, min=0.1, max=1.0, step=0.1, description="B (T)"),
         R=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description="R (Ω)"),
         l=FloatSlider(value=0.5, min=0.2, max=1.0, step=0.1, description="l (m)"),
         mass=FloatSlider(value=0.1, min=0.05, max=0.5, step=0.05, description="mass (kg)"),
         v0=FloatSlider(value=4.0, min=1.0, max=8.0, step=1.0, description="shove (m/s)"));

## A current growing through an inductor

A switch closes on a battery, a resistor and an inductor. The current cannot jump: as it rises, the inductor's back
EMF, L times the rate of change, pushes against the rise. So the current climbs, quickly at first, then levels off at
V/R, where nothing is changing and the inductor is just a wire. A bigger inductance makes the climb slower. The energy
stored in the inductor's field at the end is ½ L I².

In [ ]:
def grow_picture(L=0.5, R=5.0, V=10.0):
    tau = L / R
    run = simulate(lambda t, s: ((V - s[0] * R) / L,), (0.0,), 6 * tau, tau / 500)
    ts = np.array([t for t, _ in run]); i = np.array([s[0] for _, s in run])
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.17)
    ax.plot(ts, i, color=GOLD); ax.axhline(V / R, color=DIM, ls="--"); ax.axvline(tau, color=GHOST)
    ax.set_xlabel("time (s)"); ax.set_ylabel("current (A)")
    fig.suptitle(f"levels off at V/R = {V / R:.2f} A · L/R = {tau:.3f} s · stored in the end ½ L I² = {inductor_energy(L, V / R):.2f} J",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return i[500] / (V / R), i[-1] / (V / R)

at_tau, at_end = grow_picture(0.5, 5.0, 10.0)
assert close(at_tau, 1 - math.exp(-1), 1e-6) and at_end > 0.997                    # (V/R)(1 − e^(−tR/L))
n_steps = 10000; di = 2.0 / n_steps
assert close(sum(0.5 * (j + 0.5) * di * di for j in range(n_steps)), inductor_energy(0.5, 2.0), 1e-9)   # L i Δi, added up

In [ ]:
interact(grow_picture, L=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description="L (H)"),
         R=FloatSlider(value=5.0, min=1.0, max=20.0, step=1.0, description="R (Ω)"),
         V=FloatSlider(value=10.0, min=2.0, max=24.0, step=2.0, description="V (V)"));

## Peak and rms, and who leads whom

The rms value is the steady value with the same heating: square the wave, take the mean over a cycle, then the root.
For a sine wave that is the peak over root two; for a square wave it is the peak itself; for a triangle wave, the
peak over root three. The second picture puts a sine voltage on a part and finds the current from the part's own law,
a point at a time: in a resistor the current is in step; in an inductor it is the running sum of v/L, and it lags by a
quarter cycle; in a capacitor it is C times the slope of v, and it leads by a quarter cycle. The lead or lag is
measured on the waves, not assumed.

In [ ]:
SHAPES = {"sine": np.sin, "square": lambda x: np.sign(np.sin(x)), "triangle": lambda x: 2 / np.pi * np.arcsin(np.sin(x))}

def rms_picture(shape="sine", peak_volts=325.0):
    x = (np.arange(10000) + 0.5) * 2 * np.pi / 10000
    v = peak_volts * SHAPES[shape](x)
    r_ = float(np.sqrt(np.mean(v ** 2)))
    fig, ax = plt.subplots(figsize=(10, 3.8))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.82, bottom=0.18)
    ax.plot(np.degrees(x), v, color=CHALK); ax.axhline(r_, color=GOLD, ls="--"); ax.axhline(-r_, color=GOLD, ls="--")
    ax.set_xlabel("angle through the cycle (degrees)"); ax.set_ylabel("volts")
    fig.suptitle(f"{shape} wave: peak {peak_volts:g} V, rms {r_:.1f} V (rms/peak = {r_ / peak_volts:.4f})", x=0.03, ha="left", color=GOLD)
    footnote(fig, "10,000 points squared, averaged, rooted"); plt.show()
    return r_ / peak_volts

assert close(rms_picture("sine", 325.0), 1 / ROOT2, 1e-9) and close(rms_picture("square"), 1.0, 1e-12)
assert close(rms_picture("triangle"), 1 / math.sqrt(3), 1e-6) and agrees("P1.6", [170 * (1 / ROOT2)])

In [ ]:
interact(rms_picture, shape=Dropdown(options=list(SHAPES), value="sine", description="shape"),
         peak_volts=FloatSlider(value=325.0, min=50.0, max=400.0, step=5.0, description="peak (V)"));

In [ ]:
def lead_deg(wave, ref):
    """How far `wave` runs ahead of `ref`, in degrees (positive: it peaks earlier), both over one whole cycle."""
    n = len(wave); k = np.arange(n)
    ang = lambda w: math.atan2(float(np.sum(w * np.cos(2 * np.pi * k / n))), float(np.sum(w * np.sin(2 * np.pi * k / n))))
    return (math.degrees(ang(wave) - ang(ref)) + 180) % 360 - 180

def part_current(part, value, f=50.0, V=230.0, n=3600):
    dt = 1 / f / n; t = np.arange(n) * dt; w = omega(f)
    v = peak(V) * np.sin(w * t)
    if part == "resistor": i = v / value
    elif part == "inductor":                                     # v = L × (rate of change of i): add v/L up, step by step
        i = (np.cumsum(v) - 0.5 * v) * dt / value; i = i - i.mean()
    else:                                                        # i = C × (rate of change of v): the measured slope
        i = value * (peak(V) * np.sin(w * (t + dt)) - peak(V) * np.sin(w * (t - dt))) / (2 * dt)
    return t, v, i

def phasor_picture(part="inductor", value=0.1, hertz=50.0):
    t, v, i = part_current(part, value, hertz)
    lead = lead_deg(i, v); i_rms = float(np.sqrt(np.mean(i ** 2))); p_mean = float(np.mean(v * i))
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.09, right=0.91, top=0.82, bottom=0.17)
    ax.plot(t * 1e3, v, color=CHALK, label="voltage"); ax.set_xlabel("time (ms)"); ax.set_ylabel("volts")
    ax2 = ax.twinx(); ax2.plot(t * 1e3, i, color=SKY, label="current"); ax2.set_ylabel("amps", color=SKY); ax2.grid(False)
    word = "in step" if abs(lead) < 0.5 else (f"the current leads by {lead:.0f}°" if lead > 0 else f"the current lags by {-lead:.0f}°")
    fig.suptitle(f"{part}: {word} · {i_rms:.2f} A rms · mean power {p_mean:.1f} W", x=0.03, ha="left", color=GOLD)
    footnote(fig, "the lead or lag is measured on the waves"); plt.show()
    return lead, i_rms, p_mean

lead_L, i_L, p_L = phasor_picture("inductor", 0.1)
lead_C, i_C, p_C = phasor_picture("capacitor", 100e-6)
lead_R, i_R, p_R = phasor_picture("resistor", 100.0)
assert abs(lead_L + 90) < 0.01 and abs(lead_C - 90) < 0.01 and abs(lead_R) < 1e-9      # L lags, C leads, R in step
assert close(i_L, 230 / x_l(50, 0.1), 1e-3) and close(i_C, 230 / x_c(50, 100e-6), 1e-4) and close(p_R, 529.0, 1e-6)
assert abs(p_L) < 1e-6 and abs(p_C) < 1e-6                                             # no power, on average, in L or C

In [ ]:
interact(phasor_picture, part=Dropdown(options=["resistor", "inductor", "capacitor"], value="inductor", description="part"),
         value=FloatSlider(value=0.1, min=0.00001, max=200.0, step=0.00001, description="Ω, H or F", readout_format=".5f"),
         hertz=FloatSlider(value=50.0, min=10.0, max=500.0, step=10.0, description="f (Hz)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: a field from an EMF, a coasting rod, the charge that flows
when a coil is flipped, two coils on one tube, an inductance from a current, a power factor, a circuit designed for
a sharpness, and a wind turbine's transformer.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is one line of algebra, then one line of Python. When a coil is flipped right over, its
flux reverses, so the change is twice BA. Only a resistor takes power, so the power is I²R, the same as V I cos φ.

In [ ]:
assert agrees("P3.1", [3 * 0.05 / (250 * 40e-4)]) and agrees("P3.3", [100 * 2 * 0.2 * 10e-4 / 20 * 1e3])
M = mutual_solenoids(1500, 3000, 8e-4, 0.3)
assert agrees("P3.4", [M * 1e3, M * 2 / 0.004]) and agrees("P3.5", [230 / 0.46 / omega(50)])
Z6 = impedance(20, x_l(60, 0.1), x_c(60, 100e-6)); I6 = 230 / Z6
print(f"P3.6  Z = {Z6:.2f} Ω, cos φ = {20 / Z6:.3f}, P = I²R = {I6 ** 2 * 20:.1f} W = V I cos φ = {ac_power(230, I6, phase_deg(20, x_l(60, 0.1), x_c(60, 100e-6))):.1f} W")
assert agrees("P3.6", [20 / Z6, I6 ** 2 * 20]) and close(I6 ** 2 * 20, ac_power(230, I6, phase_deg(20, x_l(60, 0.1), x_c(60, 100e-6))), 1e-9)
C7 = 1 / (omega(1e5) ** 2 * 2e-3)
assert agrees("P3.7", [C7 * 1e9, omega(1e5) * 2e-3 / 50]) and close(q_factor(2e-3, C7, omega(1e5) * 2e-3 / 50), 50, 1e-9)
assert agrees("P3.8", [66e3 / 690, 3e6 / 66e3, line_loss(3e6, 66e3, 0.8)])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## A series LCR circuit and its resonance curve

A resistor, an inductor and a capacitor in one loop, on an alternating source. The curve is the rms current against
the source's frequency, computed from the impedance point by point. It peaks where the two reactances cancel, at
ω₀ = 1/√(LC), and there the current is V/R. The dashed line marks half the power; the width between its two crossings
is R/L, and the sharpness is Q = ω₀L/R. A bigger resistance leaves the peak where it is, but makes it lower and wider.
The gold dot is the circuit **stepped in time** at the frequency you choose, from switch-on until it settles: it lands
on the curve.

In [ ]:
def lcr_step(R, L, C, V, w, cycles=40, per=400, last=4):
    """Switch the circuit on and step it: L di/dt = v − iR − q/C. Returns the settled rms current and its lag in degrees."""
    Vm = peak(V); T = 2 * math.pi / w
    run = simulate(lambda t, s: (s[1], (Vm * math.sin(w * t) - s[1] * R - s[0] / C) / L), (0.0, 0.0), cycles * T, T / per)
    tail = run[-last * per - 1:-1]
    i = np.array([s[1] for _, s in tail]); v = np.array([Vm * math.sin(w * t) for t, _ in tail])
    k = np.arange(len(i))
    ang = lambda x: math.atan2(float(np.sum(x * np.cos(2 * np.pi * k / per))), float(np.sum(x * np.sin(2 * np.pi * k / per))))
    lag = (math.degrees(ang(v) - ang(i)) + 180) % 360 - 180
    return float(np.sqrt(np.mean(i ** 2))), lag

def resonance_picture(R=10.0, L=0.1, C_uF=10.0, V=10.0, w_step=1000.0):
    C = C_uF * 1e-6; w0 = resonant_omega(L, C)
    ws = np.linspace(0.2 * w0, 3 * w0, 4000)
    I = V / np.hypot(R, ws * L - 1 / (ws * C))
    I_s, lag_s = lcr_step(R, L, C, V, w_step)
    top = V / R / ROOT2
    inside = ws[I >= top]; width = inside[-1] - inside[0]
    fig, ax = plt.subplots(figsize=(10, 4.4))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.82, bottom=0.16)
    ax.plot(ws, I, color=CHALK); ax.axhline(top, color=DIM, ls="--"); ax.axvline(w0, color=GHOST)
    ax.plot([w_step], [I_s], "o", color=GOLD, ms=9)
    ax.set_xlabel("ω of the source (rad/s)"); ax.set_ylabel("rms current (A)")
    fig.suptitle(f"ω₀ = {w0:.0f} rad/s ({w0 / (2 * math.pi):.0f} Hz) · peak {V / R:.2f} A · width {width:.0f} rad/s · Q = {w0 * L / R:.1f}"
                 f" · stepped: {I_s:.3f} A, lag {lag_s:.0f}°", x=0.03, ha="left", color=GOLD)
    footnote(fig, "curve from the impedance · the dot stepped in time"); plt.show()
    return ws[int(np.argmax(I))], width, I_s, lag_s

w_pk, width10, I_res, lag_res = resonance_picture(10.0, 0.1, 10.0, 10.0, 1000.0)
assert close(w_pk, 1000.0, 1e-3) and close(width10, 100.0, 2e-2) and close(I_res, 1.0, 1e-3) and abs(lag_res) < 0.1
w_pk50, width50, _, _ = resonance_picture(50.0, 0.1, 10.0, 10.0, 1000.0)
assert close(w_pk50, 1000.0, 1e-3) and close(width50, 500.0, 2e-2)                 # the peak stays; five times the width
L_t, C_t = 80 / omega(50), 1 / (omega(50) * 40)                                    # the course's circuit: 30 Ω, X_L 80 Ω, X_C 40 Ω at 50 Hz
I_t, lag_t = lcr_step(30.0, L_t, C_t, 100.0, omega(50))
assert close(I_t, 2.0, 1e-3) and abs(lag_t - 53.13) < 0.05                         # 2 A, the current lagging by 53°
assert agrees("P2.9", [resonant_f(20e-3, 0.5e-6)])

In [ ]:
interact(resonance_picture, R=FloatSlider(value=10.0, min=2.0, max=100.0, step=2.0, description="R (Ω)"),
         L=FloatSlider(value=0.1, min=0.02, max=0.5, step=0.02, description="L (H)"),
         C_uF=FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0, description="C (µF)"),
         V=FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0, description="V rms"),
         w_step=FloatSlider(value=1000.0, min=300.0, max=2500.0, step=50.0, description="step at ω"));

## A power line: the same power at different voltages

A wind turbine makes 2 MW at 690 V. Send that power down a cable of half an ohm (our own example, treated as one
circuit). The current is P/V and the cable's loss is I²R = P²R/V². At 690 V the loss would be more than the turbine
makes. A transformer raises the voltage; each time the voltage is ten times bigger, the loss is a hundred times
smaller. Slide the voltage and watch the share that is lost.

In [ ]:
def line_picture(kV=33.0, MW=2.0, R=0.5):
    P = MW * 1e6
    vs = np.geomspace(0.4, 400, 200) * 1e3
    lost = np.minimum(100.0, 100 * line_loss(P, vs, R) / P)
    here = 100 * line_loss(P, kV * 1e3, R) / P
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.18)
    ax.loglog(vs / 1e3, np.maximum(lost, 1e-6), color=CHALK); ax.plot([kV], [min(100.0, here)], "o", color=GOLD, ms=9)
    ax.axvline(0.69, color=GHOST); ax.set_xlabel("line voltage (kV)"); ax.set_ylabel("share of the power lost (%)")
    fig.suptitle(f"{MW:g} MW at {kV:g} kV: {P / (kV * 1e3):,.0f} A · loss {line_loss(P, kV * 1e3, R) / 1e3:,.1f} kW ({here:.3g} %)"
                 f" · turns ratio from 690 V: {kV * 1e3 / 690:.1f}", x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · one circuit · computed"); plt.show()
    return P / (kV * 1e3), line_loss(P, kV * 1e3, R)

I_hi, loss_hi = line_picture(33.0)
I_lo, loss_lo = line_picture(0.69)
assert round(I_lo, -2) == 2900 and round(I_hi) == 61 and loss_lo > 2e6 and loss_hi < 2000
assert close(loss_lo / loss_hi, (33000 / 690) ** 2, 1e-9)                          # the loss falls as the turns ratio squared
assert close(line_picture(6.9)[1] * 100, loss_lo, 1e-9)                            # ten times the volts: a hundredth of the loss

In [ ]:
interact(line_picture, kV=FloatSlider(value=33.0, min=0.69, max=400.0, step=0.01, description="kV"),
         MW=FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="MW"),
         R=FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description="cable (Ω)"));

## Sources
Every exercise and set-up here is our own. The constant is the course's (μ₀ = 4π × 10⁻⁷ T m/A; CODATA 2022 gives the
same to nine places, physics.nist.gov/cuu/Constants). The wind turbine's figures (2 MW, 690 V, 50 Hz) are a Vestas
V90-2.0 MW's, from its general specification (2013); the cable and the single-circuit treatment are our own example,
and a real turbine is three-phase. Mains values (230 V and 50 Hz in most countries, 120 V and 60 Hz in North America)
are nominal. The metals' conductivities are typical handbook values. The simulator is the course's: fourth-order
Runge–Kutta with a fixed step.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph17_induction_ac/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph17_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (MU0, g) == (_P.MU0, _P.g)
    assert close(_P.PIPE["t_pipe"], t_cu, 1e-2) and close(_P.TURBINE["loss_high"], loss_hi, 1e-12) and close(_P.RES["w0"], w_pk, 1e-3)
    assert abs(_P.PHASE["inductor"] - lead_L) < 0.01 and abs(_P.PHASE["capacitor"] - lead_C) < 0.01
    print("matches the course's practice set")